# Data Reconciliation Tool
## Инструмент сверки данных между Oracle и Postgres (Форма 110 КХД)

**Назначение:** Выверка данных между источником (Oracle) и приемником (Postgres) для объемов 3-5 млн строк.

**Особенности:**
- Автоматическое определение типов полей
- Агрегированная сверка по ключам и числовым полям
- Гибкая фильтрация и выборка
- Выявление проблемных ключей для ручного анализа

**Исправления (v2):**
- `specific_keys` теперь принимает список словарей `[{'regn': '1000', 'datereport': '2010-02-01'}]` для поддержки составных ключей
- Нормализация регистра колонок Oracle/Postgres перед `pd.merge` (Oracle → UPPER, Postgres → lower)


## Блок 1: Импорт библиотек и настройка окружения


### 💡 Автоматическое подключение библиотек для БД из системного Python

**Ситуация:** Основные библиотеки (pandas, numpy, matplotlib, seaborn, sqlalchemy) установлены в окружении Anaconda/Jupyter, а библиотеки для работы с базами данных (`oracledb`, `psycopg2`) — в системном Python.

**Решение:** Код автоматически находит и подключает библиотеки для БД из системного Python, не требуя ручной настройки окружения.

---

In [ ]:
import sys
import os

def add_system_python_db_libraries():
    """
    Добавляет пути к библиотекам oracledb и psycopg2 из системного Python,
    если они не найдены в текущем окружении.
    """
    system_python_paths = [
        '/usr/local/lib/python3.12/site-packages',
        '/usr/local/lib/python3.11/site-packages',
        '/usr/local/lib/python3.10/site-packages',
        '/usr/lib/python3/dist-packages',
    ]
    need_oracledb = False
    need_psycopg2 = False
    try:
        import oracledb
    except ImportError:
        need_oracledb = True
    try:
        import psycopg2
    except ImportError:
        need_psycopg2 = True
    if not (need_oracledb or need_psycopg2):
        print('✓ Все библиотеки для БД уже доступны')
        return
    for sys_path in system_python_paths:
        if not os.path.exists(sys_path):
            continue
        oracledb_path = os.path.join(sys_path, 'oracledb')
        psycopg2_path = os.path.join(sys_path, 'psycopg2')
        psycopg2_binary_path = os.path.join(sys_path, 'psycopg2_binary')
        if need_oracledb and os.path.exists(oracledb_path):
            if sys_path not in sys.path:
                sys.path.insert(1, sys_path)
                print(f'✓ Добавлен путь к oracledb: {sys_path}')
            need_oracledb = False
        if need_psycopg2 and (os.path.exists(psycopg2_path) or os.path.exists(psycopg2_binary_path)):
            if sys_path not in sys.path:
                sys.path.insert(1, sys_path)
                print(f'✓ Добавлен путь к psycopg2: {sys_path}')
            need_psycopg2 = False
        if not need_oracledb and not need_psycopg2:
            break
    if need_oracledb:
        print('⚠ Предупреждение: oracledb не найден в системном Python')
    if need_psycopg2:
        print('⚠ Предупреждение: psycopg2 не найден в системном Python')

add_system_python_db_libraries()

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.pool import QueuePool
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (14, 8)

try:
    import oracledb
    print(f'✓ oracledb версии {oracledb.__version__} доступен')
    ORACLE_LIBRARY = 'oracledb'
except ImportError:
    print('✗ Библиотека oracledb не найдена. Установите: pip install oracledb')
    ORACLE_LIBRARY = None

try:
    import psycopg2
    print(f'✓ psycopg2 версии {psycopg2.__version__} доступен')
    POSTGRES_LIBRARY = 'psycopg2'
except ImportError:
    print('✗ Библиотека psycopg2 не найдена. Установите: pip install psycopg2-binary')
    POSTGRES_LIBRARY = None

print('\nБиблиотеки успешно импортированы')

## Блок 2: Класс конфигурации и основные настройки


In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Tuple

@dataclass
class ReconciliationConfig:
    oracle_connection_string: str
    postgres_connection_string: str
    oracle_schema: str
    oracle_table: str
    postgres_schema: str
    postgres_table: str
    composite_keys: List[str]
    report_date_column: Optional[str] = None
    exclusions: List[str] = field(default_factory=list)
    sample_size: Optional[int] = None
    # FIX: specific_keys теперь список словарей для поддержки составных ключей
    # Пример: [{'regn': '1000', 'datereport': '2010-02-01'}, ...]
    specific_keys: Optional[List[Dict[str, Any]]] = None
    batch_size: int = 100000
    decimal_precision: int = 2
    year_from: Optional[int] = None
    year_to: Optional[int] = None

print('Класс ReconciliationConfig определён')

## Блок 3: Основной класс DataReconciliator (агрегатный этап)


In [ ]:
class DataReconciliator:
    def __init__(self, config: ReconciliationConfig):
        self.config = config
        self.oracle_engine = None
        self.postgres_engine = None
        self.oracle_metadata = {}
        self.postgres_metadata = {}
        self.results = {}

    def connect(self):
        self.oracle_engine = create_engine(
            self.config.oracle_connection_string,
            poolclass=QueuePool,
            pool_size=2,
            max_overflow=2
        )
        self.postgres_engine = create_engine(
            self.config.postgres_connection_string,
            poolclass=QueuePool,
            pool_size=2,
            max_overflow=2
        )
        print('✓ Подключения к БД установлены')

    def disconnect(self):
        if self.oracle_engine:
            self.oracle_engine.dispose()
        if self.postgres_engine:
            self.postgres_engine.dispose()
        print('✓ Подключения закрыты')

    def _normalize_scalar(self, value, field_type):
        if value is None:
            return None
        if field_type == 'numeric':
            try:
                return round(float(value), self.config.decimal_precision)
            except (ValueError, TypeError):
                return None
        return value

    def collect_metadata(self):
        for db_type, engine, schema, table in [
            ('oracle', self.oracle_engine, self.config.oracle_schema, self.config.oracle_table),
            ('postgres', self.postgres_engine, self.config.postgres_schema, self.config.postgres_table)
        ]:
            insp = inspect(engine)
            columns = insp.get_columns(table, schema=schema)
            all_cols = [c['name'].upper() for c in columns]
            keys_upper = [k.upper() for k in self.config.composite_keys]
            excl_upper = [e.upper() for e in self.config.exclusions]
            numeric_fields = [
                c['name'].upper() for c in columns
                if str(c['type']).upper() in ('NUMBER', 'NUMERIC', 'FLOAT', 'DOUBLE', 'DECIMAL',
                                               'INTEGER', 'BIGINT', 'REAL', 'DOUBLE PRECISION')
                and c['name'].upper() not in keys_upper
                and c['name'].upper() not in excl_upper
            ]
            meta = {'all_columns': all_cols, 'classification': {
                'numeric_fields': numeric_fields,
                'key_fields': keys_upper
            }}
            if db_type == 'oracle':
                self.oracle_metadata = meta
            else:
                self.postgres_metadata = meta
        print('✓ Метаданные собраны')

    def _build_order_by_clause(self):
        keys = self.config.composite_keys
        if keys:
            return ' ORDER BY ' + ', '.join(keys)
        return ''

    def _build_aggregation_query(self, schema, table, classification, db_type):
        keys = self.config.composite_keys
        numeric_fields = classification['numeric_fields']
        key_col = ', '.join(keys)
        if db_type == 'oracle':
            select_clause = ', '.join(keys)
            aggregations = ', '.join([f'SUM({f}) AS SUM_{f}' for f in numeric_fields])
            aggregations += ', COUNT(*) AS row_cnt'
            group_by_clause = key_col
        else:
            select_clause = ', '.join([k.lower() for k in keys])
            aggregations = ', '.join([f'SUM({f.lower()}) AS SUM_{f}' for f in numeric_fields])
            aggregations += ', COUNT(*) AS row_cnt'
            group_by_clause = ', '.join([k.lower() for k in keys])
        where_clauses = []

        # FIX: поддержка составных ключей в specific_keys
        # specific_keys ожидается как список словарей:
        # [{'regn': '1000', 'datereport': '2010-02-01'}, ...]
        if self.config.specific_keys:
            row_conditions = []
            for key_dict in self.config.specific_keys:
                col_conditions = []
                for col_name, col_val in key_dict.items():
                    col_expr = col_name if db_type == 'oracle' else col_name.lower()
                    if isinstance(col_val, str):
                        col_conditions.append(f"{col_expr} = '{col_val}'")
                    else:
                        col_conditions.append(f"{col_expr} = {col_val}")
                row_conditions.append('(' + ' AND '.join(col_conditions) + ')')
            where_clauses.append('(' + ' OR '.join(row_conditions) + ')')

        if self.config.report_date_column:
            col = self.config.report_date_column if db_type == 'oracle' else self.config.report_date_column.lower()
            if self.config.year_from is not None and self.config.year_to is not None:
                if db_type == 'oracle':
                    where_clauses.append(f"EXTRACT(YEAR FROM {col}) BETWEEN {self.config.year_from} AND {self.config.year_to}")
                else:
                    where_clauses.append(f"EXTRACT(YEAR FROM {col}::DATE) BETWEEN {self.config.year_from} AND {self.config.year_to}")
            elif self.config.year_from is not None:
                if db_type == 'oracle':
                    where_clauses.append(f"EXTRACT(YEAR FROM {col}) >= {self.config.year_from}")
                else:
                    where_clauses.append(f"EXTRACT(YEAR FROM {col}::DATE) >= {self.config.year_from}")
            elif self.config.year_to is not None:
                if db_type == 'oracle':
                    where_clauses.append(f"EXTRACT(YEAR FROM {col}) <= {self.config.year_to}")
                else:
                    where_clauses.append(f"EXTRACT(YEAR FROM {col}::DATE) <= {self.config.year_to}")
        base_query = f"SELECT {select_clause}, {aggregations} FROM {schema}.{table}"
        if where_clauses:
            base_query += ' WHERE ' + ' AND '.join(where_clauses)
        base_query += f' GROUP BY {group_by_clause}'
        if self.config.sample_size is not None and self.config.sample_size > 0:
            order_by = self._build_order_by_clause()
            if db_type == 'oracle':
                query = f"SELECT * FROM ({base_query}{order_by}) WHERE ROWNUM <= {self.config.sample_size}"
            else:
                query = f"{base_query}{order_by} LIMIT {self.config.sample_size}"
        else:
            query = base_query
        return query.strip()

    def run_full_reconciliation(self):
        print('=' * 60)
        print('ЗАПУСК СВЕРКИ ДАННЫХ (АГРЕГАТНЫЙ ЭТАП)')
        print('=' * 60)
        try:
            print('\n[Шаг 1/3] Сбор метаданных таблиц...')
            self.collect_metadata()
            print('\n[Шаг 2/3] Выполнение агрегированной сверки...')
            oracle_query = self._build_aggregation_query(
                self.config.oracle_schema, self.config.oracle_table,
                self.oracle_metadata['classification'], 'oracle')
            postgres_query = self._build_aggregation_query(
                self.config.postgres_schema, self.config.postgres_table,
                self.postgres_metadata['classification'], 'postgres')
            print(f'Oracle query:\n{oracle_query}\n')
            print(f'Postgres query:\n{postgres_query}\n')
            with self.oracle_engine.connect() as conn:
                oracle_df = pd.read_sql_query(text(oracle_query), conn)
            with self.postgres_engine.connect() as conn:
                postgres_df = pd.read_sql_query(text(postgres_query), conn)

            # FIX: нормализуем регистр колонок перед merge
            # Oracle возвращает UPPER_CASE, Postgres — lower_case
            # приводим оба датафрейма к lower_case для корректного join
            oracle_df.columns = [c.lower() for c in oracle_df.columns]
            postgres_df.columns = [c.lower() for c in postgres_df.columns]

            print('\n[Шаг 3/3] Сравнение результатов по ключам...')
            keys = [k.lower() for k in self.config.composite_keys]
            merged = pd.merge(oracle_df, postgres_df, on=keys, how='outer',
                              suffixes=('_ORA', '_PG'), indicator=True)
            numeric_fields = self.oracle_metadata['classification']['numeric_fields']
            for f in numeric_fields:
                fl = f.lower()
                ora_col = f'sum_{fl}_ORA'
                pg_col = f'sum_{fl}_PG'
                if ora_col in merged.columns and pg_col in merged.columns:
                    ora_series = merged[ora_col].apply(lambda v: self._normalize_scalar(v, 'numeric'))
                    pg_series = merged[pg_col].apply(lambda v: self._normalize_scalar(v, 'numeric'))
                    merged[f'DELTA_{f}'] = [None if o is None or p is None else o - p
                                            for o, p in zip(ora_series, pg_series)]
                    merged[f'MISMATCH_{f}'] = [o != p for o, p in zip(ora_series, pg_series)]
            if 'row_cnt_ORA' in merged.columns and 'row_cnt_PG' in merged.columns:
                merged['DELTA_ROW_COUNT'] = merged['row_cnt_ORA'].fillna(0) - merged['row_cnt_PG'].fillna(0)
                merged['MISMATCH_ROW_COUNT'] = merged['row_cnt_ORA'].fillna(0) != merged['row_cnt_PG'].fillna(0)
            mismatch_flags = [c for c in merged.columns if c.startswith('MISMATCH_')]
            merged['HAS_DISCREPANCY'] = merged[mismatch_flags].any(axis=1) if mismatch_flags else False
            def assign_status(row):
                if row['_merge'] == 'left_only':
                    return 'ONLY_IN_ORACLE'
                elif row['_merge'] == 'right_only':
                    return 'ONLY_IN_POSTGRES'
                elif row['HAS_DISCREPANCY']:
                    return 'MISMATCH'
                return 'OK'
            merged['STATUS'] = merged.apply(assign_status, axis=1)
            discrepancies = merged[merged['HAS_DISCREPANCY'] | (merged['_merge'] != 'both')].copy()
            print('\n[ИТОГ] Формирование результатов агрегированной сверки...')
            self.results = {
                'stage1_merged': merged,
                'stage1_discrepancies': discrepancies,
                'summary': {
                    'total_records': len(merged),
                    'discrepancies_count': len(discrepancies),
                    'status_counts': merged['STATUS'].value_counts().to_dict(),
                },
            }
            print('\n' + '=' * 60)
            print('АГРЕГАТНАЯ СВЕРКА ЗАВЕРШЕНА УСПЕШНО')
            print('=' * 60)
            return self.results
        except Exception as e:
            print(f'\n❌ КРИТИЧЕСКАЯ ОШИБКА ПРИ ВЫПОЛНЕНИИ СВЕРКИ: {str(e)}')
            raise

print('Класс DataReconciliator обновлен (v2): исправлены specific_keys и нормализация регистра колонок')

## Блок 5: Пример вызова и тестирования


In [ ]:
# ============================================================================
# ПРИМЕР ВЫЗОВА (раскомментируйте и заполните своими данными)
# ============================================================================

# FIX: specific_keys — список словарей с ПОЛНЫМ набором ключей
# Каждый словарь описывает одну строку по всем composite_keys
# Пример фильтрации по конкретному банку и дате:
#   specific_keys=[{'regn': '1000', 'datereport': '2010-02-01'}]
#
# Несколько строк:
#   specific_keys=[
#       {'regn': '1000', 'datereport': '2010-02-01'},
#       {'regn': '2000', 'datereport': '2010-03-01'},
#   ]
#
# Без фильтрации по конкретным ключам — оставить None:
#   specific_keys=None

config = ReconciliationConfig(
    oracle_connection_string="*",
    postgres_connection_string="*",
    oracle_schema="ACTESTDWH",
    oracle_table="REPFORM101DM_FT",
    postgres_schema="rdm",
    postgres_table="repform101dm_ft",
    composite_keys=['regn', 'datereport'],
    report_date_column='datereport',
    exclusions=['A_P', 'RID', 'ID', 'LOADID', 'CHANGEFLAG'],
    sample_size=None,
    specific_keys=None,  # или [{'regn': '1000', 'datereport': '2010-02-01'}]
    batch_size=100000,
    decimal_precision=2,
    year_from=2010,
    year_to=2010,
)

reconciliator = DataReconciliator(config)
reconciliator.connect()
results = reconciliator.run_full_reconciliation()

print('Результаты агрегатной сверки (расхождения по ключам):')
if 'stage1_discrepancies' in results and len(results['stage1_discrepancies']) > 0:
    display(results['stage1_discrepancies'].head(20))
else:
    print('Нет расхождений на агрегатном уровне')

print('\nСводка:')
print(results['summary'])

reconciliator.disconnect()
print('\n🎉 Агрегатная сверка завершена!')